# 01 · 데이터셋 준비

KaggleHub에서 `a2015003713/militaryaircraftdetectiondataset` 를 내려받아  
Colab 또는 로컬의 **ephemeral cache** 안에 YOLO 학습용 데이터셋을 준비합니다.  
`labels_with_split.csv` 나 `dataset 2/` 를 수동으로 업로드할 필요가 없습니다.


## ① 파라미터

In [ ]:
from pathlib import Path

# ── 수정 가능한 파라미터 ────────────────────────────────────────────
REPO_DIR       = Path(globals().get('REPO_DIR',       '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
CACHE_ROOT     = Path(globals().get('CACHE_ROOT',     globals().get('MAD_DATA_CACHE_ROOT', '/content/.cache/mad')))
KAGGLE_DATASET_ID = globals().get('KAGGLE_DATASET_ID', 'a2015003713/militaryaircraftdetectiondataset')
OUTPUT_DIR      = Path(globals().get('OUTPUT_DIR',
                        str(CACHE_ROOT / 'datasets' / 'a2015003713__militaryaircraftdetectiondataset' / 'yolo_dataset')))
FORCE_REBUILD   = bool(globals().get('FORCE_REBUILD', False))
FORCE_DOWNLOAD  = bool(globals().get('FORCE_DOWNLOAD', False))
SMOKE_TEST      = bool(globals().get('SMOKE_TEST', False))   # True: split당 32장만 준비
SEED            = int(globals().get('SEED', 42))
# ──────────────────────────────────────────────────────────────────

print(f'KAGGLE_DATASET_ID : {KAGGLE_DATASET_ID}')
print(f'CACHE_ROOT        : {CACHE_ROOT}')
print(f'OUTPUT_DIR        : {OUTPUT_DIR}')
print(f'FORCE_REBUILD     : {FORCE_REBUILD}')
print(f'FORCE_DOWNLOAD    : {FORCE_DOWNLOAD}')
print(f'SMOKE_TEST        : {SMOKE_TEST}')


## ② 환경 초기화

In [ ]:
import sys, os
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ['MAD_WORKSPACE_ROOT'] = str(WORKSPACE_ROOT)
os.environ['MAD_DATA_CACHE_ROOT'] = str(CACHE_ROOT)

from mad.colab_utils import setup_colab_env
setup_colab_env(REPO_DIR, WORKSPACE_ROOT)

print(f'cache root: {CACHE_ROOT}')
print('Kaggle auth hint: set a Colab secret named KAGGLE_API_TOKEN if download consent/auth is required.')


## ③ YOLO 데이터셋 빌드

In [ ]:
from mad.kaggle_dataset import KaggleYOLOBuildConfig, build_kaggle_yolo_dataset

result = build_kaggle_yolo_dataset(
    KaggleYOLOBuildConfig(
        dataset_id=KAGGLE_DATASET_ID,
        output_dir=OUTPUT_DIR,
        cache_root=CACHE_ROOT,
        force_download=FORCE_DOWNLOAD,
        force_rebuild=FORCE_REBUILD,
        max_images_per_split=32 if SMOKE_TEST else None,
        seed=SEED,
        validate=True,
    )
)

print(f"\n✅ 데이터셋 준비 완료")
print(f"   dataset_yaml      : {result['dataset_yaml']}")
print(f"   raw_dataset_dir   : {result['raw_dataset_dir']}")
print(f"   class_count       : {result['class_count']}")
print(f"   processed_images  : {result['stats']['processed_images']}")
if result.get('validation'):
    v = result['validation']
    print(f"   validation_valid  : {v['valid']}")
    for sp, info in v.get('split_summary', {}).items():
        print(f"   [{sp}] images={info.get('image_count',0)}, missing_labels={info.get('missing_labels',0)}")


## ④ 빌드 요약 확인

In [ ]:
import json
from pathlib import Path

output_dir = Path(result['output_dir'])
for fname in ['build_summary.json', 'validation_summary.json']:
    fpath = output_dir / fname
    if fpath.exists():
        print(f'--- {fname} ---')
        data = json.loads(fpath.read_text(encoding='utf-8'))
        import pprint; pprint.pprint(data)
        print()

## 다음 단계

`notebooks/03_run_benchmark.ipynb` 에서 벤치마크 실험을 실행하세요.

생성된 `dataset.yaml` 경로:
```
{OUTPUT_DIR}/dataset.yaml
```